In [23]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

In [3]:
# CSV file that contains information about the tournament results historically 
tournament_results = pd.read_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Data/MNCAATourneyDetailedResults.csv')
tournament_results


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,134,1421,92,1411,84,N,1,32,69,...,31,14,31,17,28,16,15,5,0,22
1,2003,136,1112,80,1436,51,N,0,31,66,...,16,7,7,8,26,12,17,10,3,15
2,2003,136,1113,84,1272,71,N,0,31,59,...,28,14,21,20,22,11,12,2,5,18
3,2003,136,1141,79,1166,73,N,0,29,53,...,17,12,17,14,17,20,21,6,6,21
4,2003,136,1143,76,1301,74,N,1,27,64,...,21,15,20,10,26,16,14,5,8,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,2025,146,1120,70,1277,64,N,0,26,61,...,23,13,16,11,27,12,9,5,5,18
1445,2025,146,1222,69,1397,50,N,0,28,66,...,29,15,20,11,23,12,9,3,2,9
1446,2025,152,1196,79,1120,73,N,0,25,53,...,25,16,23,9,20,11,12,11,3,22
1447,2025,152,1222,70,1181,67,N,0,23,61,...,17,18,23,10,19,14,7,7,4,16


In [4]:
tournament_results.columns.tolist()

['Season',
 'DayNum',
 'WTeamID',
 'WScore',
 'LTeamID',
 'LScore',
 'WLoc',
 'NumOT',
 'WFGM',
 'WFGA',
 'WFGM3',
 'WFGA3',
 'WFTM',
 'WFTA',
 'WOR',
 'WDR',
 'WAst',
 'WTO',
 'WStl',
 'WBlk',
 'WPF',
 'LFGM',
 'LFGA',
 'LFGM3',
 'LFGA3',
 'LFTM',
 'LFTA',
 'LOR',
 'LDR',
 'LAst',
 'LTO',
 'LStl',
 'LBlk',
 'LPF']

In [5]:
# Get the seeds for those same tournament results 
slots = pd.read_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Data/MNCAATourneySlots.csv')
slots


,Season,Slot,StrongSeed,WeakSeed
0,1985,R1W1,W01,W16
1,1985,R1W2,W02,W15
2,1985,R1W3,W03,W14
3,1985,R1W4,W04,W13
4,1985,R1W5,W05,W12
...,...,...,...,...
2648,2026,R6CH,R5WX,R5YZ
2649,2026,X16,X16a,X16b
2650,2026,Y11,Y11a,Y11b
2651,2026,Y16,Y16a,Y16b


In [6]:
slots.columns.tolist()

['Season', 'Slot', 'StrongSeed', 'WeakSeed']

In [7]:
# Get the seeds 
seeds = pd.read_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Data/MNCAATourneySeeds.csv')
seeds

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
2689,2026,Z12,1219
2690,2026,Z13,1218
2691,2026,Z14,1244
2692,2026,Z15,1474


In [8]:
seeds.columns.tolist()

['Season', 'Seed', 'TeamID']

In [9]:

# Here I am bringing in the teams dataframe that contains team name information 
teams = pd.read_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Data/MTeamSpellings.csv')

# Once again, sometimes the same name may be present multiple times. To prevent this, 
# we will filter out and get the first instance of the team name 
# Get canonical names (most frequent spelling per TeamID)
canonical_names = (teams
                   .groupby('TeamID')['TeamNameSpelling']
                   .agg(lambda x: x.value_counts().index[0])
                   .reset_index()
                   .rename(columns={'TeamNameSpelling': 'TeamName'}))


# Seeds look like 'W01', 'X02', 'Y11', 'Z16' etc.
# We are only interested in the number associated with the seed code so we will strip all 
# other information other than the pure number 
seeds['SeedNum'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)
seeds['Region'] = seeds['Seed'].str[0]


# Here we want to build the tournament matchup dataframe 
# The tournament_results dataframe contains information about the historical tournament games 
tourney = tournament_results.copy()

# Here we will merge in the winning TeamID and the Seed Number using a LEFT Join 
tourney = tourney.merge(
    seeds[['Season', 'TeamID', 'SeedNum']].rename(
        columns={'TeamID': 'WTeamID', 'SeedNum': 'WSeed'}),
    on=['Season', 'WTeamID'],
    how='left'
)

#Same thing as above, here we will bring in the losing TeamID and the Seed NUmber using the LEFT JOIN 
tourney = tourney.merge(
    seeds[['Season', 'TeamID', 'SeedNum']].rename(
        columns={'TeamID': 'LTeamID', 'SeedNum': 'LSeed'}),
    on=['Season', 'LTeamID'],
    how='left'
)

# I also want to merge in the team name so that we can see a name rather than simply just a code 
# We will do this for both the winning and losing team 
tourney = tourney.merge(
    canonical_names.rename(columns={'TeamID': 'WTeamID', 'TeamName': 'WTeamName'}),
    on='WTeamID', how='left'
)
tourney = tourney.merge(
    canonical_names.rename(columns={'TeamID': 'LTeamID', 'TeamName': 'LTeamName'}),
    on='LTeamID', how='left'
)


In [10]:
# The slots table tells us exactly which round each matchup belongs to.
# R1 = Round of 64, R2 = Round of 32, R3 = Sweet 16, 
# R4 = Elite 8, R5 = Final Four, R6 = Championship

# Here we want to extract round number from Slot string
# R1-R6 are the main tournament rounds
slots['Round'] = slots['Slot'].str.extract(r'R(\d)')
slots['Round'] = slots['Round'].fillna(0).astype(int)

# To match slots to tourney games, we need the full seed strings 
# because that's how slots identifies matchups via StrongSeed and WeakSeed
tourney = tourney.merge(
    seeds[['Season', 'TeamID', 'Seed']].rename(
        columns={'TeamID': 'WTeamID', 'Seed': 'WSeedStr'}),
    on=['Season', 'WTeamID'],
    how='left'
)
tourney = tourney.merge(
    seeds[['Season', 'TeamID', 'Seed']].rename(
        columns={'TeamID': 'LTeamID', 'Seed': 'LSeedStr'}),
    on=['Season', 'LTeamID'],
    how='left'
)

# Create a consistent pairing key for matching: alphabetically sort the two seed strings
# so that the order doesn't matter when joining to slots
tourney['Seed1'] = tourney[['WSeedStr', 'LSeedStr']].min(axis=1)
tourney['Seed2'] = tourney[['WSeedStr', 'LSeedStr']].max(axis=1)

slots['Seed1'] = slots[['StrongSeed', 'WeakSeed']].min(axis=1)
slots['Seed2'] = slots[['StrongSeed', 'WeakSeed']].max(axis=1)

tourney = tourney.merge(
    slots[['Season', 'Seed1', 'Seed2', 'Round']],
    on=['Season', 'Seed1', 'Seed2'],
    how='left'
)

tourney = tourney.drop(columns=['Seed1', 'Seed2', 'WSeedStr', 'LSeedStr'])


day_to_round = {}
for d in range(132, 134): day_to_round[d] = 0   # Play-in (First Four)
for d in range(134, 136): day_to_round[d] = 1   # Round of 64
for d in range(136, 138): day_to_round[d] = 2   # Round of 32
for d in range(138, 140): day_to_round[d] = 3   # Sweet 16
for d in range(140, 142): day_to_round[d] = 4   # Elite 8
for d in range(142, 144): day_to_round[d] = 5   # Final Four
for d in range(144, 146): day_to_round[d] = 6   # Championship

tourney['RoundFromDay'] = tourney['DayNum'].map(day_to_round)
tourney['Round'] = tourney['Round'].fillna(tourney['RoundFromDay'])
tourney = tourney.drop(columns=['RoundFromDay'])

print(f"Round distribution after fix:")
print(tourney['Round'].value_counts().sort_index())
print(f"\nRemaining NaN rounds: {tourney['Round'].isna().sum()}")

Round distribution after fix:
Round
0.0     64
1.0    639
2.0     60
3.0    348
4.0      8
5.0     84
6.0    130
Name: count, dtype: int64

Remaining NaN rounds: 116


In [11]:

# Here we want to assign the favorite and the underdog status 
# The lower valued eed is the better team and the higher valued seed is the worse team 
# The FavSeed will be the the one with a lower seed 
# The UndSeed will be the one with a higher seed 
tourney['FavSeed'] = tourney[['WSeed', 'LSeed']].min(axis=1)
tourney['UndSeed'] = tourney[['WSeed', 'LSeed']].max(axis=1)

# np.where works like an IF statement:
#   np.where(condition, value_if_true, value_if_false)

# The boolean condition is: WSeed <= LSeed
#   - TRUE means the winner had the better result
#     so Winner = Favorite, Loser = Underdog
#   - FALSE means the winner had a worse seed which means an upset 
#     so Loser = Favorite, Winner = Underdog
tourney['FavTeamID'] = np.where(tourney['WSeed'] <= tourney['LSeed'], 
                                 tourney['WTeamID'], tourney['LTeamID'])

tourney['UndTeamID'] = np.where(tourney['WSeed'] <= tourney['LSeed'], 
                                 tourney['LTeamID'], tourney['WTeamID'])

tourney['FavTeamName'] = np.where(tourney['WSeed'] <= tourney['LSeed'],  
                                   tourney['WTeamName'], tourney['LTeamName'])

tourney['UndTeamName'] = np.where(tourney['WSeed'] <= tourney['LSeed'], 
                                   tourney['LTeamName'], tourney['WTeamName'])

# Here we ask if the winning team seed was less than the losing team seed to see if the 
# favorite won (otherwise upset)
# This is essentially our target variable 
tourney['FavWin'] = np.where(tourney['WSeed'] <= tourney['LSeed'], 1, 0)

# Handle same-seed matchups (first four in)
# When seeds are equal, use lower TeamID as favorite 
same_seed = tourney['WSeed'] == tourney['LSeed']
tourney.loc[same_seed, 'FavTeamID'] = tourney.loc[same_seed, ['WTeamID', 'LTeamID']].min(axis=1)
tourney.loc[same_seed, 'FavWin'] = np.where(
    tourney.loc[same_seed, 'WTeamID'] == tourney.loc[same_seed, 'FavTeamID'], 1, 0
)


In [12]:
tourney

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,WTeamName,LTeamName,Round,FavSeed,UndSeed,FavTeamID,UndTeamID,FavTeamName,UndTeamName,FavWin
0,2003,134,1421,92,1411,84,N,1,32,69,...,n.c. asheville,texas southern,0.0,16,16,1411,1411,n.c. asheville,texas southern,0
1,2003,136,1112,80,1436,51,N,0,31,66,...,arizona,vermont,1.0,1,16,1112,1436,arizona,vermont,1
2,2003,136,1113,84,1272,71,N,0,31,59,...,arizona st,memphis,1.0,7,10,1272,1113,memphis,arizona st,0
3,2003,136,1141,79,1166,73,N,0,29,53,...,c michigan,creighton,1.0,6,11,1166,1141,creighton,c michigan,0
4,2003,136,1143,76,1301,74,N,1,27,64,...,california,n.c. state,1.0,8,9,1143,1301,california,n.c. state,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1444,2025,146,1120,70,1277,64,N,0,26,61,...,auburn,michigan st,NaN,1,2,1120,1277,auburn,michigan st,1
1445,2025,146,1222,69,1397,50,N,0,28,66,...,houston,tennessee,NaN,1,2,1222,1397,houston,tennessee,1
1446,2025,152,1196,79,1120,73,N,0,25,53,...,florida,auburn,NaN,1,1,1120,1120,florida,auburn,0
1447,2025,152,1222,70,1181,67,N,0,23,61,...,houston,duke,NaN,1,1,1181,1181,houston,duke,0


In [13]:

# Seed differential: Here we want to see the difference between the underdog seed and favorite seed 
tourney['SeedDiff'] = tourney['UndSeed'] - tourney['FavSeed']

# If the favorite team won, then we paste the WScore as the FavScore
# If they did not win, then we paste the L Score
tourney['FavScore'] = np.where(tourney['FavTeamID'] == tourney['WTeamID'],
                                tourney['WScore'], tourney['LScore'])

# If the favorite team won, then we post the L Score as the UndScore and if not (underdog won)
# then we put the WScore 
tourney['UndScore'] = np.where(tourney['FavTeamID'] == tourney['WTeamID'],
                                tourney['LScore'], tourney['WScore'])

# Get the score differential 
tourney['ScoreDiff'] = tourney['FavScore'] - tourney['UndScore']


# For each game, we look at ALL prior seasons to compute:
# 1. How often does this specific seed matchup (e.g., 5 vs 12) go to the favorite?
# 2. How often does this favorite's seed win in general?
# 3. How often does this underdog's seed pull upsets in general?

# Here we sort out all tournament games chronologically so when we loop through, earlier seasons come
# first.
tourney = tourney.sort_values('Season').reset_index(drop=True)

# Empty list to store historical stats for each game 
seed_matchup_history = []

# Here we go through each tournament game one at a time and we ask three quyestions about history 
# listed above 
for idx, row in tourney.iterrows():

    # Question 1: Hoow does this exact sered matchup usually go? 
    prior = tourney[(tourney['Season'] < row['Season']) & 
                    (tourney['FavSeed'] == row['FavSeed']) & 
                    (tourney['UndSeed'] == row['UndSeed'])]
    
    # Get the historical win rate of the favorite 
    if len(prior) > 0:
        hist_winrate = prior['FavWin'].mean()
        hist_count = len(prior)
    else:
        hist_winrate = np.nan
        hist_count = 0
    
    # Question 2: How does this favorte's seed usually do? 
    prior_fav = tourney[(tourney['Season'] < row['Season']) & 
                        (tourney['FavSeed'] == row['FavSeed'])]
    
    # Get the favorite win rate for this fav seed 
    fav_rate = prior_fav['FavWin'].mean() if len(prior_fav) > 0 else np.nan
    
    # Question 3: How often does this underdog's seed pull upsets? 
    prior_und = tourney[(tourney['Season'] < row['Season']) & 
                        (tourney['UndSeed'] == row['UndSeed'])]
    
    # What is the upset rate for this seed? 
    und_upset_rate = (1 - prior_und['FavWin'].mean()) if len(prior_und) > 0 else np.nan
    
    # Get the historical win rate, favorite rate, and underdog win rate for this unique case 
    seed_matchup_history.append({
        'HistWinRate': hist_winrate,
        'HistMatchupCount': hist_count,
        'FavSeedHistWinRate': fav_rate,
        'UndSeedHistUpsetRate': und_upset_rate,
    })

hist_df = pd.DataFrame(seed_matchup_history)
tourney = pd.concat([tourney, hist_df], axis=1)

print(f"\nFavorite win rate: {tourney['FavWin'].mean():.3f}")

print(f"\nFavorite win rate by round:")
print(tourney.groupby('Round')['FavWin'].mean().round(3))



Favorite win rate: 0.699

Favorite win rate by round:
Round
0.0    0.422
1.0    0.728
2.0    0.800
3.0    0.721
4.0    0.750
5.0    0.631
6.0    0.662
Name: FavWin, dtype: float64


In [14]:
# Filter to 2016 onwards
tourney = tourney[tourney['Season'] >= 2012].copy()

print(f"Tourney shape: {tourney.shape}")
print(f"Seasons: {tourney['Season'].min()} - {tourney['Season'].max()}")
print(f"\nNaN counts per column:")
print(tourney.isna().sum())

Tourney shape: (870, 54)
Seasons: 2012 - 2025

NaN counts per column:
Season                   0
DayNum                   0
WTeamID                  0
WScore                   0
LTeamID                  0
LScore                   0
WLoc                     0
NumOT                    0
WFGM                     0
WFGA                     0
WFGM3                    0
WFGA3                    0
WFTM                     0
WFTA                     0
WOR                      0
WDR                      0
WAst                     0
WTO                      0
WStl                     0
WBlk                     0
WPF                      0
LFGM                     0
LFGA                     0
LFGM3                    0
LFGA3                    0
LFTM                     0
LFTA                     0
LOR                      0
LDR                      0
LAst                     0
LTO                      0
LStl                     0
LBlk                     0
LPF                      0
WSeed       

In [15]:
tourney.columns.tolist()

['Season',
 'DayNum',
 'WTeamID',
 'WScore',
 'LTeamID',
 'LScore',
 'WLoc',
 'NumOT',
 'WFGM',
 'WFGA',
 'WFGM3',
 'WFGA3',
 'WFTM',
 'WFTA',
 'WOR',
 'WDR',
 'WAst',
 'WTO',
 'WStl',
 'WBlk',
 'WPF',
 'LFGM',
 'LFGA',
 'LFGM3',
 'LFGA3',
 'LFTM',
 'LFTA',
 'LOR',
 'LDR',
 'LAst',
 'LTO',
 'LStl',
 'LBlk',
 'LPF',
 'WSeed',
 'LSeed',
 'WTeamName',
 'LTeamName',
 'Round',
 'FavSeed',
 'UndSeed',
 'FavTeamID',
 'UndTeamID',
 'FavTeamName',
 'UndTeamName',
 'FavWin',
 'SeedDiff',
 'FavScore',
 'UndScore',
 'ScoreDiff',
 'HistWinRate',
 'HistMatchupCount',
 'FavSeedHistWinRate',
 'UndSeedHistUpsetRate']

In [ ]:
# These are the features that the model will use to predict if the favorite wins
# We intentionally exclude any post-game information
# because at inference we won't know those yet

seed_features = [
    'FavSeed', # favorite's seed number (1-16)
    'UndSeed', # underdog's seed number (1-16)
    'SeedDiff', # gap between seeds (UndSeed - FavSeed)
    'Round', # tournament round (0=play-in, 1-6)
    'HistWinRate', # historical win rate for this exact seed matchup
    'HistMatchupCount', # how many times we've seen this matchup before
    'FavSeedHistWinRate', # how often this favorite seed wins across all matchups
    'UndSeedHistUpsetRate', # how often this underdog seed pulls upsets
]

# Define the target variable. It should be the same as the in season model
seed_target = 'FavWin'

# Get rid of rows with NaN. Does not really hurt the model in any way 
seed_model_df = tourney.dropna(subset=seed_features).copy()
seed_model_df[seed_target] = seed_model_df[seed_target].astype(int)

print(f"Total usable games: {len(seed_model_df)}")
print(f"Seasons: {seed_model_df['Season'].min()} - {seed_model_df['Season'].max()}")
print(f"Favorite win rate: {seed_model_df[seed_target].mean():.3f}")
print(f"\nGames per season:")
print(seed_model_df.groupby('Season').size())

Total usable games: 783
Seasons: 2012 - 2025
Favorite win rate: 0.695

Games per season:
Season
2012    59
2013    58
2014    61
2015    62
2016    62
2017    62
2018    59
2019    62
2021    54
2022    62
2023    61
2024    59
2025    62
dtype: int64


In [17]:

# Here I chose to do a season based split. In theory, I probably did not need 
# to because on season does not impact another, but nevertheless, I did in case
# Train: 2016-2022, Val: 2023, Test: 2024-2025
train_seasons = seed_model_df['Season'] <= 2022
val_seasons = seed_model_df['Season'] == 2023
test_seasons = seed_model_df['Season'] >= 2024

train_df = seed_model_df[train_seasons].copy()
val_df = seed_model_df[val_seasons].copy()
test_df = seed_model_df[test_seasons].copy()

print(f"\nTrain: {len(train_df)} games (Seasons {train_df['Season'].min()}-{train_df['Season'].max()})")
print(f"Val:   {len(val_df)} games (Season {val_df['Season'].min()})")
print(f"Test:  {len(test_df)} games (Seasons {test_df['Season'].min()}-{test_df['Season'].max()})")
print(f"\nTarget distribution per split:")
print(f"Train: {train_df[seed_target].mean():.3f}")
print(f"Val:   {val_df[seed_target].mean():.3f}")
print(f"Test:  {test_df[seed_target].mean():.3f}")

# Here I prepared the train, val, and test based on autogluon standards 
train_ag = train_df[seed_features + [seed_target]].copy()
val_ag = val_df[seed_features + [seed_target]].copy()
test_ag = test_df[seed_features + [seed_target]].copy()

print(f"\nTrain shape: {train_ag.shape}")
print(f"Val shape:   {val_ag.shape}")
print(f"Test shape:  {test_ag.shape}")


Train: 601 games (Seasons 2012-2022)
Val:   61 games (Season 2023)
Test:  121 games (Seasons 2024-2025)

Target distribution per split:
Train: 0.689
Val:   0.672
Test:  0.736

Train shape: (601, 9)
Val shape:   (61, 9)
Test shape:  (121, 9)


In [18]:
# Define the Predictor
seed_predictor = TabularPredictor(
    label=seed_target,
    problem_type='binary',
    eval_metric='roc_auc',
    path='mm_autogluon_model_historical'
).fit(
    train_data=train_ag,
    tuning_data=val_ag,
    time_limit=1200,
    presets='best_quality_v150',
    use_bag_holdout=True,
    num_bag_folds=8,
    num_bag_sets=3,
    num_stack_levels=2,                   
    hyperparameters={
        'XGB': {},
        'GBM': {},
        'CAT': {},
        'RF': {},
        'XT': {},
    },
    verbosity=2
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          16
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       3.60 GB / 15.84 GB (22.7%)
Disk Space Avail:   174.87 GB / 930.80 GB (18.8%)
Presets specified: ['best_quality_v150']
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=8, num_bag_sets=3
Beginning AutoGluon training ... Time limit = 1200s
AutoGluon will save models to "c:\Users\veeri\Documents\Coding_Projects\MM_2026\Notebooks\mm_autogluon_model_historical"
Train Data Rows:    601
Train Data Columns: 8
Tuning Data Rows:    61
Tuning Data Columns: 8
Label Column:       FavWin
Problem Type:       binary


In [19]:
# Leaderboard post AutoGluon training 
leaderboard = seed_predictor.leaderboard(test_ag, silent=True)
print(leaderboard.to_string(index=False))

              model  score_test  score_val eval_metric  pred_time_test  pred_time_val  fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
    CatBoost_BAG_L1    0.724895   0.589634     roc_auc        0.050879       0.264569 17.878211                 0.050879                0.264569          17.878211            1       True          3
    LightGBM_BAG_L1    0.672577   0.638415     roc_auc        0.041140       0.914859  5.596215                 0.041140                0.914859           5.596215            1       True          1
     XGBoost_BAG_L1    0.669066   0.594512     roc_auc        0.103648       1.275490  6.729977                 0.103648                1.275490           6.729977            1       True          5
    CatBoost_BAG_L2    0.647296   0.581098     roc_auc        0.411969       2.772087 48.026045                 0.071170                0.205018          16.699052            2       True          9
     

In [20]:
# Importance of Features 
importance = seed_predictor.feature_importance(test_ag)
print("\nFeature Importance:")
print(importance.to_string())

Computing feature importance via permutation shuffling for 8 features using 121 rows with 5 shuffle sets...
	4.04s	= Expected runtime (0.81s per shuffle set)
	0.55s	= Actual runtime (Completed 5 of 5 shuffle sets)



Feature Importance:
                      importance    stddev   p_value  n  p99_high   p99_low
HistWinRate             0.020927  0.026416  0.075593  5  0.075318 -0.033464
SeedDiff                0.019206  0.011240  0.009381  5  0.042349 -0.003936
FavSeed                 0.008287  0.016576  0.163116  5  0.042416 -0.025843
Round                  -0.003687  0.012872  0.721641  5  0.022816 -0.030190
UndSeed                -0.015449  0.006452  0.997066  5 -0.002165 -0.028734
HistMatchupCount       -0.021770  0.018789  0.969684  5  0.016917 -0.060456
FavSeedHistWinRate     -0.028020  0.022701  0.974575  5  0.018721 -0.074761
UndSeedHistUpsetRate   -0.058357  0.027354  0.995581  5 -0.002034 -0.114679


In [21]:
# I wanted to evaluate the model at different thresholds to find the optimum 
# threshold for inference 
y_true = test_ag[seed_target]
y_proba = seed_predictor.predict_proba(test_ag.drop(columns=[seed_target]))[1]

print("Accuracy at different confidence thresholds:")
print("=" * 55)
for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    # A game counts as "confident" if the model says either
    # P(FavWin) >= threshold OR P(FavWin) <= (1 - threshold)
    confident = (y_proba >= threshold) | (y_proba <= (1 - threshold))
    if confident.sum() > 0:
        confident_true = y_true[confident]
        confident_pred = (y_proba[confident] >= 0.5).astype(int)
        acc = (confident_true == confident_pred).mean()
        print(f"Threshold {threshold:.1f}: {confident.sum():>3} games, accuracy {acc:.3f}")

Accuracy at different confidence thresholds:
Threshold 0.5: 121 games, accuracy 0.711
Threshold 0.6:  75 games, accuracy 0.813
Threshold 0.7:  51 games, accuracy 0.824
Threshold 0.8:  25 games, accuracy 0.760
Threshold 0.9:   8 games, accuracy 1.000


In [30]:
# Confusion matrix at the threshold that I decided to use 
THRESHOLD = 0.58

y_pred = (y_proba >= THRESHOLD).astype(int)
auc = roc_auc_score(y_true, y_proba)

print(f"Threshold: {THRESHOLD}")
print(f"ROC AUC: {auc:.4f}\n")
print("Classification Report (Test Set):")
print(classification_report(y_true, y_pred, target_names=['Underdog Win', 'Favorite Win']))

# Here we want to build out our confusion matrix to see how well it did on the test set 
cm = confusion_matrix(y_true, y_pred)

fig_cm = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=['Underdog Win', 'Favorite Win'],   
        y=['Underdog Win', 'Favorite Win'],   
        text=cm,
        texttemplate="%{text}",
        textfont={"size": 16},
        colorscale="Blues",
        showscale=True,
        hovertemplate=(
            "Predicted: %{x}<br>"
            "Actual: %{y}<br>"
            "Count: %{z}<extra></extra>"
        )
    )
)

fig_cm.update_layout(
    title=f"Seed Model — Test Set Confusion Matrix (Threshold = {THRESHOLD})",
    xaxis_title="Predicted",
    yaxis_title="Actual",
    template="plotly_white",
    width=700,
    height=600
)

fig_cm.show()

Threshold: 0.58
ROC AUC: 0.6217

Classification Report (Test Set):
              precision    recall  f1-score   support

Underdog Win       0.36      0.47      0.41        32
Favorite Win       0.78      0.70      0.74        89

    accuracy                           0.64       121
   macro avg       0.57      0.58      0.57       121
weighted avg       0.67      0.64      0.65       121



In [31]:
# Same thing for the ROC plot
fpr, tpr, thresholds = roc_curve(y_true, y_proba)

# Find the ROC point corresponding to the threshold 
idx = np.argmin(np.abs(thresholds - THRESHOLD))

fig_roc = go.Figure()

# ROC curve
fig_roc.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"ROC Curve (AUC = {auc:.4f})",
        line=dict(width=3)
    )
)

# Random baseline
fig_roc.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Random (0.5)",
        line=dict(dash="dash", width=2, color="gray")
    )
)

# Threshold marker
fig_roc.add_trace(
    go.Scatter(
        x=[fpr[idx]],
        y=[tpr[idx]],
        mode="markers",
        name=f"Threshold = {THRESHOLD}",
        marker=dict(size=12, color="red"),
        hovertemplate=(
            f"Threshold: {THRESHOLD}<br>"
            "FPR: %{x:.4f}<br>"
            "TPR: %{y:.4f}<extra></extra>"
        )
    )
)

fig_roc.update_layout(
    title=f"Seed Model — Test Set ROC Curve (Threshold = {THRESHOLD})",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    width=700,
    height=600,
    legend=dict(x=0.60, y=0.05)
)

fig_roc.update_xaxes(range=[0, 1])
fig_roc.update_yaxes(range=[0, 1.05])

fig_roc.show()

In [26]:
# Here we can see some results on the actual test set 
test_results = test_df[['Season', 'FavTeamName', 'FavSeed', 'UndTeamName', 
                         'UndSeed', 'Round', 'FavWin']].copy()
test_results['Prob_FavWin'] = y_proba.values
test_results['Pred_FavWin'] = (y_proba.values >= 0.5).astype(int)
test_results['Correct'] = (test_results['FavWin'] == test_results['Pred_FavWin']).astype(int)

print(f"Overall test accuracy: {test_results['Correct'].mean():.3f}")
print(f"\nAccuracy by round:")
print(test_results.groupby('Round')['Correct'].mean().round(3))
print(f"\nSample predictions (sorted by confidence):")
print(test_results.sort_values('Prob_FavWin', ascending=False).head(20).to_string(index=False))

Overall test accuracy: 0.711

Accuracy by round:
Round
0.0    0.500
1.0    0.696
2.0    0.875
3.0    0.750
5.0    0.625
6.0    0.727
Name: Correct, dtype: float64

Sample predictions (sorted by confidence):
 Season    FavTeamName  FavSeed        UndTeamName  UndSeed  Round  FavWin  Prob_FavWin  Pred_FavWin  Correct
   2025         auburn        1         alabama st       16    2.0       1     0.915841            1        1
   2025           duke        1 mount saint mary`s       16    2.0       1     0.915841            1        1
   2025        houston        1       edwardsville       16    1.0       1     0.913775            1        1
   2025        florida        1         norfolk st       16    1.0       1     0.913775            1        1
   2024         purdue        1          grambling       16    2.0       1     0.901892            1        1
   2024 north carolina        1             wagner       16    2.0       1     0.901892            1        1
   2024    connecticut 

In [27]:
print(test_df['Round'].value_counts().sort_index())
print(test_df[test_df['Round'].isna()].shape)

Round
0.0     6
1.0    56
2.0     8
3.0    32
5.0     8
6.0    11
Name: count, dtype: int64
(0, 54)


In [28]:
# Load the processed tourney data (for historical win rate calculations)
tourney.to_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Notebooks/tourney_processed.csv', index=False)